# tools

> Measure which tools agents call, how often they fail, and where the shell stands in for a dedicated tool.

In [ ]:
#| default_exp tools

`drona-tools` is the before/after instrument for tool culling. It reads the persisted Ramabana and Leela histories and reports per-tool call and failure counts, shell commands that bypass a dedicated tool, and the same split per model.

In [ ]:
#| export
from collections import Counter, defaultdict
from datetime import datetime
from json import dumps
import re
from fastcore.xtras import Path
from fastcore.foundation import L
from fastcore.script import call_parse

from drona.core import read_history, tool_args, bypass_tool

## Report

History rows carry `at` as an epoch float, so `since` accepts an epoch or an ISO date. Old activity rows may lack `tool`, `ok`, or `args`, or carry `args` as a JSON string; the report tolerates all of them. Failure details are keyed by their first line with digits normalised so repeated errors group together.

In [ ]:
#| export
HISTORIES = L(Path.home()/f'.config/{app}/{kind}-history.jsonl' for app in ('ramabana', 'leela') for kind in ('agent', 'inline')).filter(Path.exists)

def _epoch(t):
    "`t` as seconds since the epoch, from a number, numeric string, or ISO date."
    if t in (None, ''): return None
    try: return float(t)
    except ValueError: return datetime.fromisoformat(t).timestamp()

def _detail_key(detail):
    "The first line of `detail` with digits normalised, so repeated failures group together."
    return re.sub(r'\d+', 'N', str(detail or '').partition('\n')[0])[:80]

def tool_report(
    turns,      # history turns from `read_history`
    since=None, # epoch or ISO date lower bound on `at`
    model=None, # restrict to one model name
):
    "Per-tool call and failure counts, shell bypasses, and the same split per model."
    since, calls, fails, details, bypass = _epoch(since), Counter(), Counter(), defaultdict(Counter), Counter()
    models = defaultdict(lambda: dict(calls=0, shell=0, bypass=0))
    for t in turns:
        if (since and (t.get('at') or 0) < since) or (model and t.get('model') != model): continue
        m = models[t.get('model') or '?']
        for a in t.get('activity') or []:
            tool = a.get('tool') or '?'
            calls[tool] += 1; m['calls'] += 1
            if a.get('ok') is False: fails[tool] += 1; details[tool][_detail_key(a.get('detail'))] += 1
            if tool == 'run_shell':
                m['shell'] += 1
                if b := bypass_tool(tool_args(a).get('command')): bypass[b] += 1; m['bypass'] += 1
    tools = {k: dict(calls=v, fails=fails[k], fail_rate=round(fails[k]/v, 3), failures=[[d,n] for d,n in details[k].most_common(3)])
             for k,v in calls.most_common()}
    return dict(calls=sum(calls.values()), tools=tools, bypass=dict(bypass.most_common()), models=dict(models))

def fmt_report(r):
    "`r` as a text table followed by bypass and per-model summary lines."
    w = max(map(len, r['tools']), default=4)
    rows = [f"{'tool':<{w}}  calls  fail%  top failure"]
    rows += [f"{t:<{w}}  {v['calls']:>5}  {100*v['fail_rate']:>4.0f}%  {v['failures'][0][0] if v['failures'] else ''}" for t,v in r['tools'].items()]
    rows.append(f"\n{r['calls']} calls; bypass: " + (', '.join(f'{k}={v}' for k,v in r['bypass'].items()) or 'none'))
    rows += [f"{m}: {v['calls']} calls, {v['shell']} shell, {v['bypass']} bypass" for m,v in r['models'].items()]
    return '\n'.join(rows)


In [ ]:
AT_OLD, AT_NEW, SINCE_BETWEEN = 1786431016.4, 1790392050.7, 1788000000.0
turns = [
  {'model': 'm1', 'at': AT_OLD, 'activity': [{'tool': 'run_shell', 'args': {'command': 'git diff'}, 'ok': True}]},
  {'model': 'm2', 'at': AT_NEW, 'activity': [
     {'tool': 'edit_cell', 'args': '{}', 'ok': False, 'detail': 'JSONDecodeError: Extra data at line 3'},
     {'tool': 'edit_cell', 'args': {}, 'ok': False, 'detail': 'JSONDecodeError: Extra data at line 9'},
     {'tool': 'view_file', 'ok': True}, {'ok': True}]}]
r = tool_report(turns)
assert r['calls'] == 5 and r['tools']['edit_cell']['fails'] == 2 and r['tools']['edit_cell']['fail_rate'] == 1.0
assert r['tools']['edit_cell']['failures'][0][1] == 2
assert r['tools']['edit_cell']['failures'][0][0] == 'JSONDecodeError: Extra data at line N'
assert list(r['tools']) == ['edit_cell', 'run_shell', 'view_file', '?']
assert r['tools']['run_shell']['fails'] == 0 and r['tools']['run_shell']['failures'] == []
assert r['bypass'] == {'git_diff': 1} and r['models']['m1'] == {'calls': 1, 'shell': 1, 'bypass': 1}
assert r['models']['m2'] == {'calls': 4, 'shell': 0, 'bypass': 0}
assert tool_report(turns, since=SINCE_BETWEEN)['calls'] == 4
assert tool_report(turns, since='2026-09-01')['calls'] == 4 and tool_report(turns, since=str(SINCE_BETWEEN))['calls'] == 4
assert tool_report(turns, model='m1')['calls'] == 1
assert tool_report([])['calls'] == 0 and fmt_report(tool_report([]))
assert 'edit_cell' in fmt_report(r) and 'git_diff=1' in fmt_report(r) and 'm1:' in fmt_report(r)
assert dumps(r)

## Command line

`drona-tools` reads every known history by default; pass `--history a,b` to choose, `--since` to bound the window, `--model` to pick one model, and `--json` for the machine-readable form recorded as a baseline.

In [ ]:
#| export
@call_parse
def tools_cli(
    history: str='',  # comma-separated history paths (default: every known history)
    since: str='',    # epoch or ISO date lower bound
    model: str='',    # restrict to one model
    json: bool=False, # print the report as JSON
):
    "Report tool usage, failures and shell bypasses across agent histories."
    paths = L(history.split(',')).filter() if history else HISTORIES
    r = tool_report(paths.map(read_history).concat(), since or None, model or None)
    print(dumps(r, indent=2) if json else fmt_report(r))


In [ ]:
from tempfile import TemporaryDirectory
from fastcore.xtras import Path
from json import dumps as _d
with TemporaryDirectory() as d:
    p = Path(d)/'h.jsonl'
    p.write_text('\n'.join(_d(t) for t in turns) + '\n{"partial": ')
    assert all(h.exists() for h in HISTORIES)
    tools_cli(history=str(p)); tools_cli(history=str(p), since='2026-09-01', json=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()